# Jupyter Notebook for identifying TIC stars within the continuous viewing zones around the ecliptic poles
## Employed to generate the data used for Fig. 3 of the letter

In [1]:
import pandas as pd
import os

In [2]:
def get_common_tic_stars(sector_ranges, pattern="tic/TICIDs_sector_{}.csv"):
    """
    Finds stars (by ID) that are present in all given sector ranges
    and returns their ID, RA, Dec, and Tmag.
    """

    # Store the TIC ID sets of each sector and the corresponding data frames
    id_sets = []
    dfs = []

    # Loop over all specified sector ranges
    for start, end in sector_ranges:
        for i in range(start, end + 1):
            # Construct the filename for the current sector
            file = pattern.format(i)

            # Skip missing sector files
            if not os.path.exists(file):
                print(f"Warning: missing file {file}")
                continue

            # Load TIC information for the current sector
            df = pd.read_csv(file)

            # Check that all required columns are available
            required = {"ID", "ra", "dec", "Tmag"}
            missing = required - set(df.columns)
            if missing:
                raise ValueError(f"{file} missing columns: {missing}")

            # Keep only relevant stellar parameters
            df = df[["ID", "ra", "dec", "Tmag"]]

            # Store TIC IDs and corresponding sector data
            id_sets.append(set(df["ID"].dropna()))
            dfs.append(df)

            print(f"{file}: {len(df)} stars")

    # Ensure that at least one valid sector file was loaded
    if len(id_sets) == 0:
        raise ValueError("No valid files found.")

    # Determine the intersection of TIC IDs present in all selected sectors
    common_ids = set.intersection(*id_sets)

    print("\n---------------------")
    print(f"Common IDs across selected sectors: {len(common_ids)}")

    # Keep only stars that are present in every selected sector
    filtered = [df[df["ID"].isin(common_ids)] for df in dfs]

    # Combine all sector data into a single data frame
    all_common = pd.concat(filtered, ignore_index=True)

    # Remove duplicate entries if the same TIC ID appears multiple times
    all_common = all_common.drop_duplicates(subset=["ID"])

    return all_common, common_ids

In [3]:
# Define TESS sector ranges contributing to the southern and northern continuous viewing zones (CVZs)
sector_ranges_SEP = [(1,13), (27,39), (61,69), (87,90), (93,96)]
sector_ranges_NEP = [(14,26), (40,41), (47,55), (56,60), (73,83), (84,86)]

# Identify TIC stars observed in all selected sectors of each CVZ
common_stars_SEP, common_ids_SEP = get_common_tic_stars(sector_ranges_SEP)
common_stars_NEP, common_ids_NEP = get_common_tic_stars(sector_ranges_NEP)

tic/TICIDs_sector_1.csv: 29732 stars
tic/TICIDs_sector_2.csv: 24736 stars
tic/TICIDs_sector_3.csv: 23557 stars
tic/TICIDs_sector_4.csv: 25722 stars
tic/TICIDs_sector_5.csv: 32582 stars
tic/TICIDs_sector_6.csv: 53209 stars
tic/TICIDs_sector_7.csv: 66134 stars
tic/TICIDs_sector_8.csv: 55777 stars
tic/TICIDs_sector_9.csv: 54395 stars
tic/TICIDs_sector_10.csv: 63438 stars
tic/TICIDs_sector_11.csv: 76369 stars
tic/TICIDs_sector_12.csv: 86340 stars
tic/TICIDs_sector_13.csv: 54361 stars
tic/TICIDs_sector_27.csv: 38269 stars
tic/TICIDs_sector_28.csv: 28386 stars
tic/TICIDs_sector_29.csv: 24422 stars
tic/TICIDs_sector_30.csv: 23563 stars
tic/TICIDs_sector_31.csv: 26220 stars
tic/TICIDs_sector_32.csv: 35599 stars
tic/TICIDs_sector_33.csv: 59387 stars
tic/TICIDs_sector_34.csv: 63913 stars
tic/TICIDs_sector_35.csv: 54028 stars
tic/TICIDs_sector_36.csv: 56277 stars
tic/TICIDs_sector_37.csv: 66311 stars
tic/TICIDs_sector_38.csv: 78280 stars
tic/TICIDs_sector_39.csv: 84751 stars
tic/TICIDs_sector_61.

In [4]:
# Save TIC stars common to all selected sectors of the southern CVZ
output_file_SEP = "tic/TIC_common_all_sectors_SEP.csv"
common_stars_SEP.to_csv(output_file_SEP, index=False)

# Save TIC stars common to all selected sectors of the northern CVZ
output_file_NEP = "tic/TIC_common_all_sectors_NEP.csv"
common_stars_NEP.to_csv(output_file_NEP, index=False)

# Confirm output file creation
print(f"Saved: {output_file_SEP}")
print(f"Saved: {output_file_NEP}")

Saved: tic/TIC_common_all_sectors_SEP.csv
Saved: tic/TIC_common_all_sectors_NEP.csv
